# 01 — Recolección de Datos

Este notebook descarga:
1. **Precios históricos** de 5–10 mercados de Polymarket via CLOB API (datos diarios).
2. **Noticias globales** de GDELT para keywords relacionadas con cada mercado.

Salidas:
- `data/raw/prices_<slug>.csv`  → serie diaria por mercado
- `data/raw/news_<slug>.csv`   → artículos GDELT por mercado
- `data/raw/markets_catalog.csv` → catálogo completo de mercados seleccionados

In [9]:
import sys
import os
from pathlib import Path

# Add project root to path so `src` is importable
ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Raw data dir:", RAW_DIR)

Project root: C:\Users\oscar\repos\predikt
Raw data dir: C:\Users\oscar\repos\predikt\data\raw


In [2]:
import pandas as pd
from datetime import datetime, timezone, timedelta

from src.polymarket import PolymarketClient
from src.gdelt_news import GdeltNewsClient

## 1. Catálogo de mercados (Gamma API)

Listamos mercados **cerrados** con alto volumen y seleccionamos los 8 más grandes.
La API devuelve entre otros el campo `clobTokenIds` que necesitaremos para el CLOB API.

In [10]:
SPORTS_KEYWORDS = [
    "super bowl", "nfl", "nba", "mlb", "nhl", "ufc", "fifa",
    "championship", "world cup", "stanley cup", "bowl", "playoffs",
    "win the", "golden boot", "grand slam", "wimbledon", "formula 1", "nascar",
    "premier league", "champions league",
]

def is_sports(question: str) -> bool:
    return any(kw in question.lower() for kw in SPORTS_KEYWORDS)


pm = PolymarketClient(request_delay=0.4)

# Fetch a large pool of recently-closed markets (Feb-Apr 2026)
all_markets = pm.list_markets(
    closed=True, active=False, limit=200,
    end_date_min="2026-02-01", end_date_max="2026-04-24",
    min_volume=5_000,
)
print(f"Mercados totales (pool): {len(all_markets)}")

# Filter out sports markets (these have extreme UP/DOWN imbalance)
markets = (
    all_markets[~all_markets["question"].apply(is_sports)]
    .nlargest(10, "volume")
    .reset_index(drop=True)
)
print(f"Mercados no deportivos seleccionados: {len(markets)}")
markets[["slug", "question", "volume", "startDate", "endDate"]]

Mercados totales (pool): 176
Mercados no deportivos seleccionados: 10


,slug,question,volume,startDate,endDate
0,us-recession-in-2025,US recession in 2025?,1.173451e+07,2025-01-08T01:33:54.924Z,2026-02-28T12:00:00Z
1,will-gta-6-cost-100,Will GTA 6 cost $100+?,8.069449e+06,2025-03-06T20:51:01.626726Z,2026-02-28T12:00:00Z
2,will-the-us-collect-between-500b-and-1t-in-rev...,Will the U.S. collect between $500b and $1t in...,2.670721e+06,2025-04-15T22:24:07.509458Z,2026-02-28T12:00:00Z
3,will-the-us-collect-less-than-100b-in-revenue-...,Will the U.S. collect less than $100b in reven...,2.306902e+06,2025-04-15T22:23:02.177852Z,2026-02-28T12:00:00Z
4,will-the-us-collect-between-100b-and-200b-in-r...,Will the U.S. collect between $100b and $200b ...,1.736957e+06,2025-04-15T22:23:23.737338Z,2026-02-28T12:00:00Z
5,will-tariffs-generate-250b-in-2025,Will tariffs generate >$250b in 2025?,1.412916e+06,2025-04-15T22:17:42.099593Z,2026-02-28T12:00:00Z
6,will-iphone-17-cost-1000-or-more,"Will iPhone 17 cost $1,000 or more?",1.326859e+06,2025-04-08T19:34:51.658Z,2026-02-28T12:00:00Z
7,will-the-us-collect-between-200b-and-500b-in-r...,Will the U.S. collect between $200b and $500b ...,1.116195e+06,2025-04-15T22:23:51.621015Z,2026-02-28T12:00:00Z
8,will-iphone-17-cost-1500-or-more,"Will iPhone 17 cost $1,500 or more?",9.551405e+05,2025-04-08T22:09:23.107213Z,2026-02-28T12:00:00Z
9,bitboy-convicted,BitBoy convicted?,8.366216e+05,2025-03-26T16:49:31.084Z,2026-03-31T12:00:00Z


In [11]:
# Save catalog
catalog_path = RAW_DIR / "markets_catalog.csv"
markets.to_csv(catalog_path, index=False)
print(f"Catálogo guardado en {catalog_path}")

Catálogo guardado en C:\Users\oscar\repos\predikt\data\raw\markets_catalog.csv


## 2. Precios históricos (CLOB API)

Para cada mercado extraemos la serie de precios del outcome YES (índice 0 de `clobTokenIds`).
El precio oscila entre 0 y 1, representando la probabilidad implícita del mercado.

In [12]:
# Remove stale CSVs from previous runs so old markets don't carry over
for f in RAW_DIR.glob("prices_*.csv"):
    f.unlink()
for f in RAW_DIR.glob("news_*.csv"):
    f.unlink()
print("Datos anteriores eliminados.")

price_frames = {}

for _, row in markets.iterrows():
    slug = row["slug"]
    try:
        df = pm.fetch_market_prices(row, outcome_index=0)
        if df.empty:
            print(f"[SKIP] {slug} — sin datos de precio")
            continue
        out_path = RAW_DIR / f"prices_{slug}.csv"
        df.to_csv(out_path, index=False)
        price_frames[slug] = df
        print(f"[OK] {slug} — {len(df)} días | {df['date'].min().date()} → {df['date'].max().date()}")
    except Exception as e:
        print(f"[ERROR] {slug}: {e}")

print(f"\nTotal mercados con precios: {len(price_frames)}")

Datos anteriores eliminados.
[OK] us-recession-in-2025 — 358 días | 2025-01-09 → 2026-01-01
[OK] will-gta-6-cost-100 — 360 días | 2025-03-07 → 2026-03-01
[OK] will-the-us-collect-between-500b-and-1t-in-revenue-in-2025 — 320 días | 2025-04-16 → 2026-03-01
[OK] will-the-us-collect-less-than-100b-in-revenue-in-2025 — 315 días | 2025-04-16 → 2026-02-28


Request failed for https://clob.polymarket.com/prices-history: HTTPSConnectionPool(host='clob.polymarket.com', port=443): Read timed out. (read timeout=30)


[ERROR] will-the-us-collect-between-100b-and-200b-in-revenue-in-2025: HTTPSConnectionPool(host='clob.polymarket.com', port=443): Read timed out. (read timeout=30)
[OK] will-tariffs-generate-250b-in-2025 — 320 días | 2025-04-16 → 2026-03-01
[OK] will-iphone-17-cost-1000-or-more — 158 días | 2025-04-09 → 2025-09-13
[OK] will-the-us-collect-between-200b-and-500b-in-revenue-in-2025 — 320 días | 2025-04-16 → 2026-03-01
[OK] will-iphone-17-cost-1500-or-more — 158 días | 2025-04-09 → 2025-09-13
[OK] bitboy-convicted — 369 días | 2025-03-27 → 2026-04-01

Total mercados con precios: 9


## 3. Noticias (GDELT Doc 2.0 API)

Para cada mercado construimos una keyword de búsqueda basada en el título.
GDELT cubre los últimos ~90 días. Para mercados más antiguos usaremos BigQuery (ver README).

In [14]:
# Define date window: last 3 months from today
END_DATE = datetime.now(tz=timezone.utc)
START_DATE = END_DATE - timedelta(days=89)  # GDELT rolling window ~90 days

start_str = START_DATE.strftime("%Y-%m-%d")
end_str   = END_DATE.strftime("%Y-%m-%d")
print(f"Ventana de noticias: {start_str} → {end_str}")

Ventana de noticias: 2026-01-25 → 2026-04-24


In [15]:
def make_keyword(question: str, max_words: int = 4) -> str:
    """Extract the most informative words from a market question as search keyword."""
    # Remove question marks and common filler
    import re
    skip = {"will", "the", "a", "an", "in", "be", "to", "of", "or", "and", "for", "at", "is"}
    tokens = re.sub(r"[^a-zA-Z0-9 ]", "", question.lower()).split()
    tokens = [t for t in tokens if t not in skip and len(t) > 2]
    return " ".join(tokens[:max_words])


gdelt = GdeltNewsClient(request_delay=1.2)
news_frames = {}

for _, row in markets.iterrows():
    slug = row["slug"]
    keyword = make_keyword(row["question"])
    print(f"Buscando: '{keyword}' para [{slug}]")
    try:
        articles = gdelt.search_articles(keyword, start_str, end_str)
        out_path = RAW_DIR / f"news_{slug}.csv"
        articles.to_csv(out_path, index=False)
        news_frames[slug] = articles
        print(f"{len(articles)} artículos guardados en {out_path.name}")
    except Exception as e:
        print(f"ERROR: {e}")

Buscando: 'recession 2025' para [us-recession-in-2025]
38 artículos guardados en news_us-recession-in-2025.csv
Buscando: 'gta cost 100' para [will-gta-6-cost-100]


GDELT search failed for keyword='gta cost 100': HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Max retries exceeded with url: /api/v2/doc/doc?query=%22gta%20cost%20100%22%20&startdatetime=20260125000000&enddatetime=20260424000000&maxrecords=250&mode=artlist&format=json (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.gdeltproject.org', port=443) at 0x19b2a993750>, 'Connection to api.gdeltproject.org timed out. (connect timeout=None)'))


0 artículos guardados en news_will-gta-6-cost-100.csv
Buscando: 'collect between 500b revenue' para [will-the-us-collect-between-500b-and-1t-in-revenue-in-2025]


No articles found for keyword='collect between 500b revenue' between 2026-01-25 and 2026-04-24


0 artículos guardados en news_will-the-us-collect-between-500b-and-1t-in-revenue-in-2025.csv
Buscando: 'collect less than 100b' para [will-the-us-collect-less-than-100b-in-revenue-in-2025]


...

0 artículos guardados en news_will-the-us-collect-between-200b-and-500b-in-revenue-in-2025.csv
Buscando: 'iphone cost 1500 more' para [will-iphone-17-cost-1500-or-more]


GDELT search failed for keyword='iphone cost 1500 more': 


0 artículos guardados en news_will-iphone-17-cost-1500-or-more.csv
Buscando: 'bitboy convicted' para [bitboy-convicted]


GDELT search failed for keyword='bitboy convicted': 


0 artículos guardados en news_bitboy-convicted.csv


## 4. Resumen de recolección

In [16]:
summary = []
for slug in price_frames:
    n_prices = len(price_frames[slug])
    n_articles = len(news_frames.get(slug, pd.DataFrame()))
    q = markets.loc[markets["slug"] == slug, "question"].values[0]
    summary.append({"slug": slug, "question": q[:60], "n_prices": n_prices, "n_articles": n_articles})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# Save summary
(RAW_DIR / "collection_summary.csv").write_text(summary_df.to_csv(index=False))
print("\nResumen guardado.")

                                                        slug                                                     question  n_prices  n_articles
                                        us-recession-in-2025                                        US recession in 2025?       358          38
                                         will-gta-6-cost-100                                       Will GTA 6 cost $100+?       360           0
  will-the-us-collect-between-500b-and-1t-in-revenue-in-2025 Will the U.S. collect between $500b and $1t in revenue in 20       320           0
       will-the-us-collect-less-than-100b-in-revenue-in-2025    Will the U.S. collect less than $100b in revenue in 2025?       315           0
                          will-tariffs-generate-250b-in-2025                        Will tariffs generate >$250b in 2025?       320           0
                            will-iphone-17-cost-1000-or-more                         Will iPhone 17 cost $1,000 or more?        158     

Como se puede observar, la recolección de datos externos a los proporcionados por la API de Polymarket fue pobre, por lo que lo pospusimos para la siguiente entrega.